# FinSurvival Competition: AFT Model Starter Notebook

**Objective:** This notebook provides a complete workflow for creating a valid submission using the `WeibullAFTFitter` model.

**Submission Requirement:** Your final submission must be a `.zip` file containing:
1. A `preprocessing.py` file.
2. A `model.py` file containing your `Model` class (this notebook will handle the renaming).
3. 16 separate pre-trained model files (`.pkl`), one for each task.

## Step 1: Setup and Imports

In [20]:
# !pip install pandas numpy lifelines==0.27.8 scikit-learn==1.2.2 joblib==1.2.0 scikit-survival==0.21.0

In [1]:
# Install required packages
# pip install -q pandas lifelines==0.27.8 scikit-learn==1.2.2 joblib==1.2.0 scikit-survival==0.21.0

# Import libraries
import pandas as pd
import joblib
import os
import shutil
from lifelines import WeibullAFTFitter
from lifelines.exceptions import ConvergenceError

# Import your custom functions and classes from their respective files
from preprocessing import preprocess
from aft_model import Model
from survival_metrics import get_concordance_index

## Step 2: Define Datasets

In [2]:
# Define all 16 event pairs for the competition
index_events = ["Borrow", "Deposit", "Repay", "Withdraw"]
outcome_events = index_events + ["Liquidated"]

event_pairs = []
for index_event in index_events:
    for outcome_event in outcome_events:
        if index_event == outcome_event:
            continue
        event_pairs.append((index_event, outcome_event))

## Step 3: Loop, Train, Evaluate, and Save Models

In [6]:
# BASE_DATA_PATH should be the path to the folder containing the training data.
BASE_DATA_PATH = '../data/input_data/'
SUBMISSION_DIR = 'my_submission_aft'
os.makedirs(SUBMISSION_DIR, exist_ok=True)

training_scores = {}

for index_event, outcome_event in event_pairs:
    print(f"\n{'='*50}")
    print(f"Processing dataset: {index_event} -> {outcome_event}")
    print(f"{'='*50}")
    
    dataset_path = os.path.join(index_event, outcome_event)
    
    # --- 1. Load Training Data ---
    print("Loading data...")
    try:
        train_df = pd.read_csv(os.path.join(BASE_DATA_PATH, dataset_path, 'train.csv'))
    except FileNotFoundError as e:
        print(f"Training data for {dataset_path} not found. Skipping. Details: {e}")
        continue
        
    # --- 2. Preprocess Data ---
    print("Preprocessing data...")
    X_train, y_train, _ = preprocess(train_df)

    # --- 3. Train Model ---
    try:
        print("Training model on full training data...")
        lifelines_train_df = pd.concat([X_train, y_train.reset_index(drop=True)], axis=1)
        
        lifelines_train_df = lifelines_train_df.loc[lifelines_train_df['timeDiff'] > 0].copy()
        
        # Instantiate and fit the Weibull AFT model
        model = WeibullAFTFitter(penalizer=0.1)
        model.fit(lifelines_train_df, duration_col='timeDiff', event_col='status')
        
        # --- 4. Evaluate on Training Data ---
        print("Evaluating model on training data...")
        X_train_eval = lifelines_train_df.drop(columns=['timeDiff', 'status'])
        y_train_eval = lifelines_train_df[['timeDiff', 'status']]
        
        wrapped_model_for_eval = Model(model)
        train_preds = wrapped_model_for_eval.predict(X_train_eval)
        c_index_train = get_concordance_index(y_train_eval, train_preds.values)
        training_scores[dataset_path] = c_index_train
        print(f"  -> Training C-Index: {c_index_train:.4f}")

        # --- 5. Save the Wrapped Model for Submission ---
        final_model_to_save = Model(model)
        model_filename = dataset_path.replace(os.sep, '_') + '.pkl'
        model_save_path = os.path.join(SUBMISSION_DIR, model_filename)
        joblib.dump(final_model_to_save, model_save_path)
        print(f"  -> Model saved to {model_save_path}")
        
    except (ConvergenceError, ValueError) as e:
        print(f"\nERROR: The model for {dataset_path} failed to train. Skipping.")
        print(f"Details: {e}")

print("\n\nAll models have been trained and saved.")


Processing dataset: Borrow -> Deposit
Loading data...
Training data for Borrow/Deposit not found. Skipping. Details: [Errno 2] No such file or directory: '../data/input_data/Borrow/Deposit/train.csv'

Processing dataset: Borrow -> Repay
Loading data...
Training data for Borrow/Repay not found. Skipping. Details: [Errno 2] No such file or directory: '../data/input_data/Borrow/Repay/train.csv'

Processing dataset: Borrow -> Withdraw
Loading data...
Training data for Borrow/Withdraw not found. Skipping. Details: [Errno 2] No such file or directory: '../data/input_data/Borrow/Withdraw/train.csv'

Processing dataset: Borrow -> Liquidated
Loading data...
Training data for Borrow/Liquidated not found. Skipping. Details: [Errno 2] No such file or directory: '../data/input_data/Borrow/Liquidated/train.csv'

Processing dataset: Deposit -> Borrow
Loading data...
Training data for Deposit/Borrow not found. Skipping. Details: [Errno 2] No such file or directory: '../data/input_data/Deposit/Borrow/

## Step 4: Training Set Score Summary

In [7]:
print("--- Training Set Scores Summary ---")
for dataset, score in training_scores.items():
    print(f"{dataset:<25} C-Index: {score:.4f}")

--- Training Set Scores Summary ---


## Step 5: Create the Submission ZIP file

In [8]:
# Copy necessary python files into the submission folder, renaming the model file
shutil.copy('preprocessing.py', SUBMISSION_DIR)
shutil.copy('aft_model.py', os.path.join(SUBMISSION_DIR, 'model.py'))

# The name of the output zip file
output_zip_filename = 'submission_aft'

# Create the zip file
shutil.make_archive(output_zip_filename, 'zip', SUBMISSION_DIR)

print(f"Successfully created '{output_zip_filename}.zip' from the '{SUBMISSION_DIR}' directory.")

Successfully created 'submission_aft.zip' from the 'my_submission_aft' directory.
